In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
#======= Read in Performance Metrics Files =======
summary = pd.read_csv('../../../Xe-Kr/Data_Size_Test/Summary_metrics.csv')
# actual initial size (n_iters counts the first fit): PX 35 -> 36, PXT 15 -> 20 and 30 -> 40
summary['n_init'] = (summary.n_train_final_mean - summary.n_iters_mean + 1).round().astype(int)
# PXT 15 = 20 and 30 = 40 are the same run: keep one
summary = summary.drop_duplicates(['run', 'space', 'n_init']).sort_values('n_init')
# standard error of the mean over the 5 data splits
summary['MAPE_se'] = summary.MAPE_std / np.sqrt(summary.n_seeds)
summary['n_iters_mean'] = summary.n_iters_mean.round().astype(int)

def metrics(run, space):
    # same column order as the old Metrics files (size, R2, MAE, MAPE, iterations) + MAPE standard error
    d = summary[(summary.run == run) & (summary.space == space)]
    return d[['n_init', 'R2_mean', 'MAE_mean', 'MAPE_mean', 'n_iters_mean', 'MAPE_se']].reset_index(drop=True)

In [ ]:
#======= Define Plot Values and Generate Plots ========
# P-X Dataframes
xe_px = metrics('TAL_Pure_Xe_Xe', 'px')
print(xe_px)
kr_px = metrics('TAL_Pure_Kr_Kr', 'px')

# P-X-T Dataframes
xe_pxt = metrics('TAL_Pure_Xe_Xe', 'pxt')
kr_pxt = metrics('TAL_Pure_Kr_Kr', 'pxt')

# P-X Values
data_size_px_kr = kr_px.iloc[:, 0]
data_size_px_xe = xe_px.iloc[:, 0]

# Xe metrics
mape_xe_px = xe_px.iloc[:, 3]
no_iters_xe_px = xe_px.iloc[:, 4]

# Kr metrics
mape_kr_px = kr_px.iloc[:, 3]
no_iters_kr_px = kr_px.iloc[:, 4]

# P-X-T Values
data_size_pxt_kr = kr_pxt.iloc[:, 0]
data_size_pxt_xe = xe_pxt.iloc[:, 0]

# Kr metrics
mape_kr_pxt = kr_pxt.iloc[:, 3]
no_iters_kr_pxt = kr_pxt.iloc[:, 4]

# Xe metrics
mape_xe_pxt = xe_pxt.iloc[:, 3]
no_iters_xe_pxt = xe_pxt.iloc[:, 4]

# Standard Error Values
se_kr_px = kr_px.iloc[:, 5]
se_xe_px = xe_px.iloc[:, 5]

se_kr_pxt = kr_pxt.iloc[:, 5]
se_xe_pxt = xe_pxt.iloc[:, 5]

# Generate Plot
panel_labels = ["(a)", "(b)"]
fig, ax = plt.subplots(figsize=(15, 6), ncols=2)

scale = 3  # adjust bubble size

# P-X Plot
for i, ds in enumerate(data_size_px_kr):
    ax[0].scatter(ds, mape_kr_px[i]*100, s=no_iters_kr_px[i] * scale, 
               color='steelblue', alpha=0.5, zorder=5, edgecolors = 'darkblue')
for i, ds in enumerate(data_size_px_xe):
    ax[0].scatter(ds, mape_xe_px[i]*100, s=no_iters_xe_px[i] * scale, 
               color='darkgreen', alpha=0.5, zorder=5, edgecolors = 'black')

# Connect bubbles with a thin line to show trend
ax[0].plot(data_size_px_kr, mape_kr_px*100, color='steelblue',  linestyle='--', linewidth=2.0, label='Kr')
ax[0].plot(data_size_px_xe, mape_xe_px*100, color='darkgreen', linestyle='--', linewidth=2.0, label='Xe')

ax[0].errorbar(data_size_px_kr, mape_kr_px*100, yerr=se_kr_px*100, fmt='none', ecolor='darkblue',
               elinewidth=1.5, capsize=4, capthick=1.5, zorder=10)
ax[0].errorbar(data_size_px_xe, mape_xe_px*100, yerr=se_xe_px*100, fmt='none', ecolor='black',
               elinewidth=1.5, capsize=4, capthick=1.5, zorder=10)
ax[0].margins(x=0.06, y=0.10)

# Size legend
for iters in [min(no_iters_xe_px), max(no_iters_kr_px)]:
    ax[0].scatter([], [], s=iters * scale, color='grey', alpha=0.5,
               label=f'{iters} TAL iters')

ax[0].set_xlabel('Initial Training Data Size', fontsize = 15, fontweight="bold")
ax[0].set_ylabel('MAPE (%)', fontsize = 15, fontweight="bold")
for tick in ax[0].get_xticklabels() + ax[0].get_yticklabels():
    tick.set_fontsize(13)
    tick.set_fontweight("bold")
# ax[0].set_title('TAL Performance vs Starting Mixture Data Size - P-X Space', fontsize = 15)
ax[0].legend(loc='upper right', fontsize = 12)
ax[0].grid(True, linestyle='--', alpha=0.4)
ax[0].text(-0.10,1.02,panel_labels[0],transform=ax[0].transAxes,
        fontsize=14,
        fontweight="bold",
        va="top",
        ha="left",
        clip_on=False)

# P-X-T Plot
for i, ds in enumerate(data_size_pxt_kr):
    ax[1].scatter(ds, mape_kr_pxt[i]*100, s=no_iters_kr_pxt[i] * scale, 
               color='steelblue', alpha=0.5, zorder=5, edgecolors = 'darkblue')
for i, ds in enumerate(data_size_pxt_xe):
    ax[1].scatter(ds, mape_xe_pxt[i]*100, s=no_iters_xe_pxt[i] * scale, 
               color='darkgreen', alpha=0.5, zorder=5, edgecolors = 'black')

# Connect bubbles with a thin line to show trend
ax[1].plot(data_size_pxt_kr, mape_kr_pxt*100, color='steelblue',  linestyle='--', linewidth=2.0, label='Kr')
ax[1].plot(data_size_pxt_xe, mape_xe_pxt*100, color='darkgreen', linestyle='--', linewidth=2.0, label='Xe')

ax[1].errorbar(data_size_pxt_kr, mape_kr_pxt*100, yerr=se_kr_pxt*100, fmt='none', ecolor='darkblue',
               elinewidth=1.5, capsize=4, capthick=1.5, zorder=10)
ax[1].errorbar(data_size_pxt_xe, mape_xe_pxt*100, yerr=se_xe_pxt*100, fmt='none', ecolor='black',
               elinewidth=1.5, capsize=4, capthick=1.5, zorder=10)
ax[1].margins(x=0.06, y=0.10)

# Size legend
for iters in [min(no_iters_xe_pxt), max(no_iters_kr_pxt)]:
    ax[1].scatter([], [], s=iters * scale, color='grey', alpha=0.5,
               label=f'{iters} TAL iters')

ax[1].set_xlabel('Initial Training Data Size', fontsize = 15, fontweight="bold")
ax[1].set_ylabel('MAPE (%)', fontsize = 15, fontweight="bold")
for tick in ax[1].get_xticklabels() + ax[1].get_yticklabels():
    tick.set_fontsize(13)
    tick.set_fontweight("bold")
# ax[1].set_title('TAL Performance vs Starting Mixture Data Size - P-X-T Space', fontsize = 15)
ax[1].legend(loc='best', fontsize = 12)
ax[1].grid(True, linestyle='--', alpha=0.4)
ax[1].text(-0.10,1.02,panel_labels[1],transform=ax[1].transAxes,
        fontsize=14,
        fontweight="bold",
        va="top",
        ha="left",
        clip_on=False)
plt.tight_layout()
plt.savefig('Data_size_test.png', dpi=600, bbox_inches="tight", pad_inches=0.05, facecolor="white")
plt.savefig('Data_size_test.pdf', dpi=600, bbox_inches="tight", pad_inches=0.05, facecolor="white")
plt.show()